In [2]:
import os
import librosa
import numpy as np
import tensorflow as tf
from tensorflow.image import resize
from sklearn.model_selection import train_test_split

## Multi Input CNN Architecture

#### In this setup, your RTX 3050 will process four parallel branches. Each branch extracts features from one stem, and then they are all "glued" together into a single brain.

In [4]:
# Load the bundle
data = np.load('indian_genre_multi_input_data.npz')

# Extract the arrays
X_vocal = data['vocal']
X_drums = data['drums']
X_bass  = data['bass']
X_other = data['other']
y       = data['labels']

print(f"Dataset loaded: {X_vocal.shape} aligned samples ready for training.")

Dataset loaded: (10198, 150, 150, 1) aligned samples ready for training.


### 3 way data split

In [5]:
# 1. Create a master index list based on the number of samples
indices = np.arange(X_vocal.shape[0])

# 2. First Split: Separate 80% for Training and 20% for (Val + Test)
train_idx, temp_idx = train_test_split(indices, test_size=0.2, random_state=42, stratify=y)

# 3. Second Split: Split the 20% into 10% Validation and 10% Testing
val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, random_state=42, stratify=y[temp_idx])

# --- APPLY THE SPLIT TO ALL STEMS ---
# Training Set
X_train = [X_vocal[train_idx], X_drums[train_idx], X_bass[train_idx], X_other[train_idx]]
y_train = y[train_idx]

# Validation Set (Used during model.fit)
X_val = [X_vocal[val_idx], X_drums[val_idx], X_bass[val_idx], X_other[val_idx]]
y_val = y[val_idx]

# Test Set (Used only at the very end for final evaluation)
X_test = [X_vocal[test_idx], X_drums[test_idx], X_bass[test_idx], X_other[test_idx]]
y_test = y[test_idx]

print(f"Training samples:   {len(train_idx)}")
print(f"Validation samples: {len(val_idx)}")
print(f"Testing samples:    {len(test_idx)}")

#important stratify
# In Indian music datasets, some genres might have fewer songs than others. Using stratify=y ensures that your Train, Val, and Test sets
# all have the exact same percentage of Bollypop, Semiclassical, etc. This prevents the model from being biased toward a specific genre
# just because it saw it more during training.

#shuffle also during training process - true be defautl

Training samples:   8158
Validation samples: 1020
Testing samples:    1020


In [6]:
import gc

# 1. Delete the original massive arrays from the .npz load
# (Replace these names with the exact variables you used to load the data)
del X_vocal
del X_drums
del X_bass
del X_other
del y
# If you created a temporary index array, delete that too
del indices 

# 2. Force the Garbage Collector to reclaim the RAM
gc.collect()

print("Original arrays deleted. System RAM cleared for training.")

Original arrays deleted. System RAM cleared for training.


In [17]:
indices = np.arange(X_vocal.shape[0]) #making a list of indices for the vocals, drums, other and bass separation
indices

train_idx, temp_idx = train_test_split(indices, test_size=0.2, random_state=42, stratify=y)

train_idx # indices randomize automatically by the train test split

# here for example
X_vocal.shape[0] # total 10198 mels

# will pick mel by randomized indices calculated above like below:

X_vocal[train_idx]

# or

y[train_idx] # both the mel' and their label are still synchronized for all vocals, drums, others and bass
# cause the random generated indices are same for all stems

# Even though the indices are "shuffled" (e.g., instead of 0,1,2,3, they might be 45,12,802,3), the relative position is preserved across all your variables.

# If train_idx[0] is index 45:

# X_vocal[45] is the Vocal for Song X.

# X_drums[45] is the Drums for Song X.

# y[45] is the Label for Song X.

# Because you use the same 45 for every stem, the "Vocals" of Song X will never accidentally be paired with the "Drums" of Song Y.

array([4, 2, 2, ..., 1, 1, 4])

### Data Split Visualization or see how it got divided

In [7]:
print(X_train[0].shape) # shape of vocal  # Xtrain list of 4 stems each of 80% of total
print(X_train[1].shape) # shape of drums
print(X_train[2].shape) # shape of bass
print(X_train[3].shape) # shape of other

print()

print(X_val[0].shape) # shape of vocal  # # Xval list of 4 stems each of 20% of total
print(X_val[1].shape) # shape of drums
print(X_val[2].shape) # shape of bass
print(X_val[3].shape) # shape of other

print()

print(X_test[0].shape) # shape of vocal  # Xtest list of 4 stems each of 20% of total
print(X_train[1].shape) # shape of drums
print(X_test[2].shape) # shape of bass
print(X_test[3].shape) # shape of other

print(y.shape)

(8158, 150, 150, 1)
(8158, 150, 150, 1)
(8158, 150, 150, 1)
(8158, 150, 150, 1)

(1020, 150, 150, 1)
(1020, 150, 150, 1)
(1020, 150, 150, 1)
(1020, 150, 150, 1)

(1020, 150, 150, 1)
(8158, 150, 150, 1)
(1020, 150, 150, 1)
(1020, 150, 150, 1)


NameError: name 'y' is not defined

# Data Normalization

## 3. Important: Normalize Separately or Together?
There is a small but vital detail here: Do not normalize using the Global Max of all 4 stems.

The Reason: "Drums" and "Vocals" have very different physical properties. If you use one "Global Max," a quiet stem like "Bass" might end up scaled to 0.0 to 0.1, while "Vocals" get 0.0 to 1.0.

The Fix: My function above normalizes each stem independently. This ensures the "Drums" branch sees the full 0–1 range of drum energy, and the "Vocals" branch sees the full 0–1 range of vocal energy.

In [8]:
def normalize_stems(stem_list):
    normalized_list = []
    for stem_array in stem_list:
        # Get min and max for this specific stem batch
        min_val = np.min(stem_array)
        max_val = np.max(stem_array)
        
        # Scale to 0-1 range
        # Formula: (x - min) / (max - min)
        normalized_stem = (stem_array - min_val) / (max_val - min_val)
        normalized_list.append(normalized_stem)
        
    return normalized_list

# Apply to your split data
X_train_norm = normalize_stems(X_train)
X_val_norm   = normalize_stems(X_val)
X_test_norm  = normalize_stems(X_test)

print("Normalization complete. New range:", np.min(X_train_norm[0]), "to", np.max(X_train_norm[0]))

Normalization complete. New range: 0.0 to 1.0


In [9]:
# Add this inside your normalization function or right after
# Normalizing creates floats (decimal numbers). In NumPy, these are often float64 by default, which take up twice as much RAM as float32.
# The Optimization: Convert your normalized arrays to float32 immediately to save 50% of your VRAM.

X_train_norm = [s.astype('float32') for s in X_train_norm]
X_val_norm   = [s.astype('float32') for s in X_val_norm]
X_test_norm  = [s.astype('float32') for s in X_test_norm]   

#float 64 takes too much system can crash almost 100%

In [7]:
## Your Final Data CheckAt this point,
#  you have:Aligned stems.
# Shuffled and Stratified splits
# Normalized values ($0.0$ to $1.0$).
# Memory-optimized (float32).

In [8]:
### Not a number(NaN) or inf or -inf value check
def check_data_integrity(stem_list, names=["Vocal", "Drums", "Bass", "Other"]):
    for i, stem in enumerate(stem_list):
        has_nan = np.isnan(stem).any()
        has_inf = np.isinf(stem).any()
        
        print(f"--- {names[i]} Integrity ---")
        print(f"  Contains NaN: {has_nan}")
        print(f"  Contains Inf: {has_inf}")
        
        if has_nan or has_inf:
            print(f"  ⚠️ WARNING: {names[i]} stem has invalid values!")
        else:
            print(f"  ✅ {names[i]} stem is clean.")

# Run the check on your normalized training data
check_data_integrity(X_train_norm)

--- Vocal Integrity ---
  Contains NaN: False
  Contains Inf: False
  ✅ Vocal stem is clean.
--- Drums Integrity ---
  Contains NaN: False
  Contains Inf: False
  ✅ Drums stem is clean.
--- Bass Integrity ---
  Contains NaN: False
  Contains Inf: False
  ✅ Bass stem is clean.
--- Other Integrity ---
  Contains NaN: False
  Contains Inf: False
  ✅ Other stem is clean.


### 3. Final Step: One-Hot Encoding Labels

In [9]:
## 3. Final Step: One-Hot Encoding Labels
# Since you have 5 unique labels ([0, 1, 2, 3, 4]), you have two choices for your loss function:

# Sparse Categorical Crossentropy: Keep labels as integers (0, 1, 2...).

# Categorical Crossentropy: Convert labels to "One-Hot" vectors (e.g., [1, 0, 0, 0, 0]).

## Moving to Training
Now that your X_train, X_val, and X_test lists are ready, your data is officially "Clean." You have:

Aligned Stems (All 4 stems match the same 4 seconds).

Balanced Classes (Thanks to stratify=y).

No Ordering Bias (Thanks to the shuffle).

In [9]:
import tensorflow as tf
from tensorflow.keras import layers, models

def build_multi_input_model(input_shape=(150, 150, 1), num_classes=5):
    # 1. Define the 4 Inputs (Synchronized Tunnels)
    in_vocal = layers.Input(shape=input_shape, name="vocal_in")
    in_other = layers.Input(shape=input_shape, name="other_in")
    in_drum  = layers.Input(shape=input_shape, name="drum_in")
    in_bass  = layers.Input(shape=input_shape, name="bass_in")

    # 2. Optimized Feature Extractor (Mini-CNN)
    def extract_features(x):
        x = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(x)
        x = layers.BatchNormalization()(x) # Helps with normalized data
        x = layers.MaxPooling2D((2, 2))(x)
        
        x = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(x)
        x = layers.BatchNormalization()(x)
        x = layers.GlobalAveragePooling2D()(x) # RAM efficient for RTX 3050
        return x

    # 3. Process all 4 stems in parallel
    v_feat = extract_features(in_vocal)
    o_feat = extract_features(in_other)
    d_feat = extract_features(in_drum)
    b_feat = extract_features(in_bass)

    # 4. Concatenate (Relationship Learning)
    # Total features: 64 * 4 = 256
    merged = layers.Concatenate()([v_feat, o_feat, d_feat, b_feat])

    # 5. Final Dense Layers
    x = layers.Dense(128, activation='relu')(merged)
    x = layers.Dropout(0.5)(x) # Prevents overfitting
    
    # CRITICAL: Changed from 2 to num_classes (5)
    output = layers.Dense(num_classes, activation='softmax')(x) 

    model = models.Model(inputs=[in_vocal, in_other, in_drum, in_bass], outputs=output)
    
    # Using 'sparse' because your y_train are integers [0,1,2,3,4]
    model.compile(optimizer='adam', 
                  loss='sparse_categorical_crossentropy', 
                  metrics=['accuracy'])
    
    return model

In [10]:
# Initialize the model
multi_model = build_multi_input_model(input_shape=(150, 150, 1), num_classes=5)
# multi_model.summary()

In [11]:
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

# 1. Save only the BEST version of the model
checkpoint = ModelCheckpoint(
    'best_indian_genre_model.keras', 
    monitor='val_accuracy', 
    save_best_only=True, 
    mode='max'
)

# 2. Stop early if accuracy stops improving (saves electricity and GPU life)
early_stop = EarlyStopping(
    monitor='val_loss', 
    patience=5, 
    restore_best_weights=True
)

callbacks_list = [checkpoint, early_stop]

In [12]:
# Assuming X_train_norm is your list: [v_train, o_train, d_train, b_train]
history = multi_model.fit(
    x=X_train_norm,
    y=y_train,
    validation_data=(X_val_norm, y_val),
    epochs=50,
    batch_size=4, 
    # callbacks=callbacks_list, # Make sure this includes your ModelCheckpoint
    verbose=1
)

# # Visualize the result
# plot_training_history(history)

Epoch 1/50
2040/2040 [==============================] - 46s 20ms/step - loss: 1.3657 - accuracy: 0.4425 - val_loss: 1.3543 - val_accuracy: 0.4735
Epoch 2/50
2040/2040 [==============================] - 39s 19ms/step - loss: 1.1416 - accuracy: 0.5331 - val_loss: 1.2749 - val_accuracy: 0.5000
Epoch 3/50
2040/2040 [==============================] - 38s 19ms/step - loss: 1.0637 - accuracy: 0.5699 - val_loss: 0.8133 - val_accuracy: 0.6882
Epoch 4/50
2040/2040 [==============================] - 38s 19ms/step - loss: 0.9919 - accuracy: 0.6047 - val_loss: 1.5863 - val_accuracy: 0.5108
Epoch 5/50
2040/2040 [==============================] - 38s 19ms/step - loss: 0.9190 - accuracy: 0.6377 - val_loss: 0.8887 - val_accuracy: 0.6529
Epoch 6/50
2040/2040 [==============================] - 38s 19ms/step - loss: 0.8548 - accuracy: 0.6679 - val_loss: 0.9792 - val_accuracy: 0.6176
Epoch 7/50
2040/2040 [==============================] - 38s 19ms/step - loss: 0.8368 - accuracy: 0.6731 - val_loss: 0.7294 -

In [14]:
# Save in the modern Keras format
multi_model.save('indian_music_classifier_multi_CNN_v1.h5') 

In [30]:
# big float value error of lr in the last
import json
import numpy as np

class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.floating):
            return float(obj)
        if isinstance(obj, np.integer):
            return int(obj)
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        return super(NumpyEncoder, self).default(obj)

# Now saving is a one-liner
with open('training_hist_multi_CNN_v1.json', 'w') as f:
    json.dump(history.history, f, cls=NumpyEncoder, indent=4)

In [31]:
training_loss ,training_accuracy = multi_model.evaluate(X_train_norm, y_train)

InternalError: Failed copying input tensor from /job:localhost/replica:0/task:0/device:CPU:0 to /job:localhost/replica:0/task:0/device:GPU:0 in order to run _EagerConst: Dst tensor is not initialized.

In [15]:
model = tf.keras.models.load_model("indian_music_classifier_multi_CNN_v1.h5")
model.summary()

Model: "model_1"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 vocal_in (InputLayer)          [(None, 150, 150, 1  0           []                               
                                )]                                                                
                                                                                                  
 other_in (InputLayer)          [(None, 150, 150, 1  0           []                               
                                )]                                                                
                                                                                                  
 drum_in (InputLayer)           [(None, 150, 150, 1  0           []                               
                                )]                                                          

In [16]:
training_loss ,training_accuracy = model.evaluate(X_test_norm, y_test)

InternalError: Failed copying input tensor from /job:localhost/replica:0/task:0/device:CPU:0 to /job:localhost/replica:0/task:0/device:GPU:0 in order to run _EagerConst: Dst tensor is not initialized.